# Merge Context Features into Main Features

This notebook merges **context saturation metrics** (from `getting_context_features.ipynb`) into the **main features file**. The result adds columns for with-context and no-context metrics (mid/last layer) so you can compare xRAG vs non-xRAG tokens.

**Inputs:** (1) Context features JSONL (with `mid_context_metrics`, `last_context_metrics`, `mid_no_context_metrics`, `last_no_context_metrics`). (2) Main features JSONL (from your feature pipeline). Rows are matched by `id`. **Output:** One JSONL with main features plus flattened context metric columns.


In [1]:
import json
import pandas as pd

# Paths (set before running; use BASE_PATH for consistency)
BASE_PATH = "/app/xlong/scripts/data_preprocessing/runs/squad" 
CONTEXT_FEATURES_PATH = f"{BASE_PATH}/context_features.jsonl"   # from getting_context_features.ipynb
MAIN_FEATURES_PATH = f"{BASE_PATH}/probe/features.jsonl"         # base features (e.g. probing pipeline)
OUTPUT_MERGED_PATH = f"{BASE_PATH}/merged_features.jsonl"       # output

In [2]:
# =========================
# FLATTEN NESTED DICTS -> NUMERIC FEATURE COLUMNS
# =========================

def safe_to_dict(x):
    if isinstance(x, dict):
        return x
    if pd.isna(x):
        return {}
    return {}


def flatten_dict_column(df: pd.DataFrame, col: str, prefix: str) -> pd.DataFrame:
    """
    Expands df[col] which contains dicts into columns: f"{prefix}.{k}".
    """
    if col not in df.columns:
        return df
    expanded = df[col].apply(safe_to_dict).apply(pd.Series)
    expanded = expanded.add_prefix(prefix + ".")
    return pd.concat([df.drop(columns=[col]), expanded], axis=1)

In [3]:
# Load context features (mid/last with context and no-context metrics)
rows = []
with open(CONTEXT_FEATURES_PATH, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))
df_context = pd.DataFrame(rows)
print(f"Context features: {len(df_context)} rows")

Context features: 100 rows


In [4]:
# Optional: inspect context columns
df_context.columns.tolist()

['id',
 'question',
 'gold_answer',
 'task_type',
 'mid_context_metrics',
 'last_context_metrics',
 'mid_no_context_metrics',
 'last_no_context_metrics']

In [5]:
# Load main features (base pipeline output; must have matching 'id')
rows = []
with open(MAIN_FEATURES_PATH, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))
df_merged = pd.DataFrame(rows)
print(f"Main features: {len(df_merged)} rows")

Main features: 100 rows


In [6]:
# Merge: add context metric columns from df_context (matched by id) and flatten dicts
context_cols = ["mid_context_metrics", "last_context_metrics", "mid_no_context_metrics", "last_no_context_metrics"]
cols_to_merge = ["id"] + [c for c in context_cols if c in df_context.columns]
df_merged = df_merged.merge(df_context[cols_to_merge], on="id", how="left")
for col in context_cols:
    if col in df_merged.columns:
        df_merged = flatten_dict_column(df_merged, col, prefix=col)
print("After merge and flatten:", df_merged.shape)

After merge and flatten: (100, 64)


In [7]:
# Save merged features
df_merged.to_json(OUTPUT_MERGED_PATH, orient="records", lines=True)
print(f"Saved: {OUTPUT_MERGED_PATH}")

Saved: /app/xlong/scripts/data_preprocessing/runs/squad/merged_features.jsonl
